# Breast Cancer Model Training

This notebook trains all five required classification models for ML Assignment 2 and saves the trained models as `.pkl` files.

## 1. Import Libraries

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

## 2. Set Project Paths

In [ ]:
current_path = Path.cwd()
project_root = current_path.parent if current_path.name == "model" else current_path

data_path = project_root / "data.csv"
model_dir = project_root / "model"
model_dir.mkdir(exist_ok=True)

print("Project root:", project_root)
print("Data path:", data_path)
print("Model folder:", model_dir)

## 3. Define Helper Functions

In [ ]:
def load_data(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    unnamed_cols = [c for c in df.columns if str(c).lower().startswith("unnamed")]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)

    return df


def build_models() -> dict:
    models = {
        "Logistic Regression": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=2000, random_state=42)),
            ]
        ),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "KNN": Pipeline(
            steps=[("scaler", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=5))]
        ),
        "Naive Bayes": Pipeline(steps=[("scaler", StandardScaler()), ("model", GaussianNB())]),
        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced",
        ),
    }
    return models


def evaluate_model(model, x_test, y_test) -> dict:
    y_pred = model.predict(x_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(x_test)[:, 1]
    else:
        y_prob = y_pred

    return {
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "AUC": round(roc_auc_score(y_test, y_prob), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
    }

## 4. Load Dataset and Prepare Train/Test Split

In [ ]:
df = load_data(data_path)

# Convert target label: M = 1, B = 0
y = df["diagnosis"].map({"M": 1, "B": 0}).astype(int)
x = df.drop(columns=["id", "diagnosis"])

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Total records:", len(df))
print("Training records:", len(x_train))
print("Test records:", len(x_test))
print("Features used:", len(x.columns))

## 5. Train Models, Evaluate, and Save Outputs

In [ ]:
models = build_models()
rows = []

for model_name, model in models.items():
    model.fit(x_train, y_train)
    metrics = evaluate_model(model, x_test, y_test)

    rows.append({"Model": model_name, **metrics})

    out_path = model_dir / f"{model_name.lower().replace(' ', '_')}.pkl"
    joblib.dump(model, out_path)

metrics_df = pd.DataFrame(rows).sort_values(by="Accuracy", ascending=False)
metrics_df.to_csv(project_root / "model_metrics.csv", index=False)

test_df = x_test.copy()
test_df["diagnosis"] = np.where(y_test.values == 1, "M", "B")
test_df.to_csv(project_root / "test_data.csv", index=False)

with open(model_dir / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(list(x.columns), f, indent=2)

metrics_df

## 6. Classification Reports and Confusion Matrices

Run this section to display the required evaluation evidence for the assignment PDF.

In [ ]:
for model_name, model in models.items():
    y_pred = model.predict(x_test)

    print("=" * 70)
    print(f"Classification Report: {model_name}")
    print("=" * 70)
    print(classification_report(y_test, y_pred, target_names=["Benign", "Malignant"]))

    fig, ax = plt.subplots(figsize=(4.5, 3.5))
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        labels=[0, 1],
        display_labels=["Benign", "Malignant"],
        cmap="Greens",
        ax=ax,
    )
    ax.set_title(f"Confusion Matrix - {model_name}")
    plt.tight_layout()
    plt.show()

## 7. Saved Files

After running this notebook, the following files are generated or updated:

- `model/logistic_regression.pkl`
- `model/decision_tree.pkl`
- `model/knn.pkl`
- `model/naive_bayes.pkl`
- `model/random_forest.pkl`
- `model/feature_columns.json`
- `model_metrics.csv`
- `test_data.csv`